# 1. Data Cleaning

Given a DataFrame like:

id	name	age	city
1	Alice	25	NY
2	Bob	null	LA
2	Bob	null	LA
3	null	40	NY

Ask the candidate to:

- Remove duplicates
- Fill missing ages with the average age
- Replace missing names with "Unknown"
- Filter people over 30

Skills tested:

dropDuplicates()
fillna()
aggregations
filtering

In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as func
from pyspark.sql.functions import udf



spark = SparkSession.builder.appName("DataCleaningExample").getOrCreate()
sc = spark.sparkContext

people = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./interview-prep-data/data-cleaning-example.csv")
)

# remove duplicates based on user_id
cleaned_duplicates = people.dropDuplicates(["id"])

#fill missing ages with average
avg_age = cleaned_duplicates.agg(func.avg("age")).collect()[0][0]
cleaned_missing_ages = cleaned_duplicates.fillna({"age": avg_age})

#fill missing names with "Unknown"
fill_val = "Unknown"
cleaned_missing_names = cleaned_missing_ages.fillna({"name": fill_val})

#filter out people over 30
final_df = cleaned_missing_names.filter(cleaned_missing_names.age <= 30)

#output to check correctness
final_df.write.format("csv") \
    .option("header", True) \
    .mode("overwrite") \
    .save("./interview-prep-data/output-data-cleaning-example.csv")

#close session
spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/09 10:34:26 WARN Utils: Your hostname, MerielesPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/09 10:34:26 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/09 10:34:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


# 2. Group By and Aggregation

Sales data:

customer	product	amount
A	Phone	500
A	Laptop	1000
B	Phone	700

Questions:

- Total spend per customer
- Average purchase amount
- Highest spending customer

Skills:

- groupBy
- agg
- sum
- avg
- orderBy

In [1]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as func
from pyspark.sql.functions import udf
from prettytable import PrettyTable

#spark: start!
spark = SparkSession.builder.appName("GroupByExample").getOrCreate()
sc = spark.sparkContext

#make our data frame
sales = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./interview-prep-data/group-by-example.csv")
)

#clean out missing values
sales = sales.filter(func.col("amount").isNotNull())
sales = sales.filter(func.col("product").isNotNull())

#get total spend per customer
total_spend = sales.groupBy("customer").agg(func.sum("amount").alias("total_spend"))

#get avg puchase amount
avg_purchase_amt = total_spend.agg(func.avg("total_spend").alias("avg_spend"))

#order by highest paying customer
total_spend = total_spend.orderBy(func.col("total_spend").desc())

#outputs
print("total spend per customer:")
total_spend.show()
avg_purchase_amt.show()

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/09 11:34:50 WARN Utils: Your hostname, MerielesPC, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/09 11:34:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/09 11:34:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


total spend per customer:
+-------------+-----------+
|     customer|total_spend|
+-------------+-----------+
|     Beta Ltd|    10900.0|
|  Epsilon LLC|     8775.0|
|     Delta Co|     8150.0|
|   Alpha Corp|     7300.0|
|    Gamma Inc|     4775.0|
|Zeta Partners|     3425.0|
+-------------+-----------+

+-----------------+
|        avg_spend|
+-----------------+
|7220.833333333333|
+-----------------+



# 3. Top N per Group

Example:

department	employee	salary
IT	John	100
IT	Mary	200
IT	Alex	150

Return the highest-paid employee in each department.

Expected solution:

Use a Window function.

Tests:

- Window.partitionBy
- row_number
- rank
- dense_rank

In [7]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as func
from pyspark.sql import Window

spark = SparkSession.builder.appName("WindowFunctionsExample").getOrCreate()
sc = spark.sparkContext

#data frame creation
employees = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./interview-prep-data/window-example.csv")
)

#test partition
windowSpec = Window.partitionBy("department").orderBy(func.col("salary").desc())
employees = employees.withColumn("rank", func.rank().over(windowSpec))
employees.filter(employees.rank == 1).show()

spark.stop()

+-----------+--------+------+----+
| department|employee|salary|rank|
+-----------+--------+------+----+
|Engineering| Matthew|141962|   1|
|    Finance|  Robert|120478|   1|
|         HR| William| 99866|   1|
|  Marketing|  Olivia|102817|   1|
|      Sales|  Harper| 88749|   1|
+-----------+--------+------+----+



# 4. Joins

Customers

id	name
1	Alice
2	Bob

Orders

customer_id	amount
1	100
1	50
3	25

Questions:

- Inner join
- Left join
- Customers with no orders
- Total spend per customer

Tests:

- joins
- null handling
- aggregations

In [13]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("JoinExample").getOrCreate()
sc = spark.sparkContext

#import customers data
customers = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./interview-prep-data/joins-users-example.csv")
)

#import transactions data
transactions = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./interview-prep-data/joins-transactions-example.csv")
)

#inner join + output
inner_join = transactions.join(customers, transactions.customer_id == customers.id, "inner")
inner_join.show()

#left join + output
left_join = customers.join(transactions, customers.id == transactions.customer_id, "left")
left_join.show()

#identify customers with no orders
print("customers with no orders")
left_join.filter(left_join.amount.isNull()).show()

#identify total spend per customer
print("total spend by customer id")
total_spend = transactions.groupBy("customer_id").agg(func.sum("amount").alias("total_spend"))
total_spend.show()

spark.stop()

+-----------+------+---+-------+
|customer_id|amount| id|   name|
+-----------+------+---+-------+
|          1|   250|  1|  Alice|
|          2|   700|  2|    Bob|
|          3|   400|  3|Charlie|
|          3|   150|  3|Charlie|
|          4|  1200|  4|  David|
+-----------+------+---+-------+

+---+-------+-----------+------+
| id|   name|customer_id|amount|
+---+-------+-----------+------+
|  1|  Alice|          1|   250|
|  2|    Bob|          2|   700|
|  3|Charlie|          3|   400|
|  3|Charlie|          3|   150|
|  4|  David|          4|  1200|
|  5|   Emma|       NULL|  NULL|
|  7|  Grace|       NULL|  NULL|
|  8|  Henry|       NULL|  NULL|
+---+-------+-----------+------+

customers with no orders
+---+-----+-----------+------+
| id| name|customer_id|amount|
+---+-----+-----------+------+
|  5| Emma|       NULL|  NULL|
|  7|Grace|       NULL|  NULL|
|  8|Henry|       NULL|  NULL|
+---+-----+-----------+------+

total spend by customer id
+-----------+-----------+
|customer

# 5. Find Duplicate Records

Dataset:

email
a@test.com
b@test.com
a@test.com
c@test.com

Tasks:

- Find duplicate emails
- Count occurrences
- Keep only unique rows

In [26]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("emailexample").getOrCreate()
sc = spark.sparkContext

emails = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./interview-prep-data/email-example.csv")
)

emails = emails.withColumn('lowercase', func.lower("email"))
emails.dropDuplicates(['lowercase']).show()


+--------------------+--------------------+
|               email|           lowercase|
+--------------------+--------------------+
|         @domain.com|         @domain.com|
|   ADMIN@COMPANY.ORG|   admin@company.org|
|alex.jones@gmail.com|alex.jones@gmail.com|
|   b_smith@yahoo.com|   b_smith@yahoo.com|
|clara.daniels@out...|clara.daniels@out...|
|            contact@|            contact@|
| developer1@test.com| developer1@test.com|
|    hello@startup.io|    hello@startup.io|
|    hr_dept@firm.com|    hr_dept@firm.com|
|    info@webshop.biz|    info@webshop.biz|
|jane.smith@yahoo.com|jane.smith@yahoo.com|
|  John.Doe@gmail.com|  john.doe@gmail.com|
|marketing_team@co...|marketing_team@co...|
|missing_at_symbol...|missing_at_symbol...|
|    sales@enterprise|    sales@enterprise|
| support@service.net| support@service.net|
|user.test@domain.com|user.test@domain.com|
+--------------------+--------------------+



# 6. Window Function Challenge

Transactions

user	date	amount
A	Jan1	10
A	Jan2	20
A	Jan3	15

Questions:

- Running total
- Previous transaction
- Difference from previous transaction
- Rolling 7-day average

Tests:

- lag
- lead
- cumulative sums
- window frames

In [ ]:
import sys
from pyspark.sql import SparkSession
from pyspark.sql import StructType, StructField, StringType, IntegerType
from pyspark.sql import functions as func

spark = SparkSession.builder.appName("window-func").getOrCreate()
sc = spark.sparkContext

#schema set
schema = StructType([
    StructField('user', 
                StringType(), True),
    StructField('date',
                StringType(), True),
    StructField('amount',
                IntegerType(), True)
])

#import csv
transactions = spark.read.schema(schema).option("header", True).csv("./interview-prep-data/6-example.csv")


